In [ ]:
import os
import librosa
import numpy as np
from scipy.fft import fft
import pandas as pd

def fft_analysis(mp3_klasor_yolu):
    sonuclar = []
    for dosya in os.listdir(mp3_klasor_yolu):
        if dosya.endswith(".mp3"):
            dosya_yolu = os.path.join(mp3_klasor_yolu, dosya)

            # Ses verisini yükle ve örnekleme frekansını al
            ses_verisi, ornekleme_sikligi = librosa.load(dosya_yolu, sr=None, mono=True)

            # Örnekleme işlemi (örneklem faktörü)
            n = 100
            orneklenmis_ses = ses_verisi[::n]

            # FFT uygula
            fft_sonucu = fft(orneklenmis_ses)

            # FFT sonucunu frekans alanında temsil etmek için gerekli frekans dizisini oluştur
            frekanslar = np.fft.fftfreq(len(orneklenmis_ses), d=1/ornekleme_sikligi)

            # Genellikle sadece pozitif frekanslarla çalışılır (çift sayıda örnek aldığımız için)
            pozitif_frekanslar = frekanslar[:len(frekanslar)//2]
            pozitif_fft = fft_sonucu[:len(frekanslar)//2]

            # FFT sonucunu 5 parçaya böl
            parca_sayisi = 5
            frekans_parcalari = np.array_split(pozitif_frekanslar, parca_sayisi)
            fft_parcalari = np.array_split(np.abs(pozitif_fft), parca_sayisi)

            # Her parçanın ortalama, varyans, standart sapma, minimum ve maksimumunu hesapla
            ortalamalar = []
            varyanslar = []
            standart_sapmalar = []
            minimumlar = []
            maksimumlar = []
            for i, frekans_parcalari in enumerate(frekans_parcalari):
                parca_fft = fft_parcalari[i]
                ortalama = np.mean(parca_fft)
                varyans = np.var(parca_fft)
                standart_sapma = np.std(parca_fft)
                minimum = np.min(parca_fft)
                maksimum = np.max(parca_fft)
                ortalamalar.append(ortalama)
                varyanslar.append(varyans)
                standart_sapmalar.append(standart_sapma)
                minimumlar.append(minimum)
                maksimumlar.append(maksimum)

            sonuc = {
                "Dosya": dosya,
                **{f"Ortalama{i+1}": ortalamalar[i] for i in range(parca_sayisi)},
                **{f"Varyans{i+1}": varyanslar[i] for i in range(parca_sayisi)},
                **{f"Standart Sapma{i+1}": standart_sapmalar[i] for i in range(parca_sayisi)},
                **{f"Minimum{i+1}": minimumlar[i] for i in range(parca_sayisi)},
                **{f"Maksimum{i+1}": maksimumlar[i] for i in range(parca_sayisi)}
            }
            sonuclar.append(sonuc)

    # Sonuçları bir DataFrame'e dönüştür
    sonuclar_df = pd.DataFrame(sonuclar)

    # DataFrame'i CSV dosyasına yaz
    sonuclar_df.to_csv('fft_analiz.csv', index=False)
    print("CSV dosyası oluşturuldu.")

# Mp3 dosyalarının bulunduğu klasör yolu
mp3_klasor_yolu = 'mp3_tracks'

# FFT analizi uygula ve sonuçları CSV dosyasına aktar
fft_analysis(mp3_klasor_yolu)
